In [71]:
import pandas as pd
import os
import json
import numpy as np
from os.path import dirname

root_path = dirname(os.getcwd())

pd.set_option("display.max_columns", None)
data_dir_processed = root_path + "/data/datasets/processed/"
data_dir_graphs = root_path + "/data/datasets/graphs/"

print(root_path, data_dir_processed, data_dir_graphs, sep="\n")

/home/sebastiano.dissegna/SephigraphV2
/home/sebastiano.dissegna/SephigraphV2/data/datasets/processed/
/home/sebastiano.dissegna/SephigraphV2/data/datasets/graphs/


In [72]:
with open("dataset_features.json", 'r') as file:
    datasets_info = json.load(file)


In [73]:
list(datasets_info.keys())

['bpi_2012', 'bpi_2013', 'sp2020', 'BPI20_RequestForPayment']

In [74]:
dataset = "BPI20_RequestForPayment"

In [75]:
tab_all = pd.read_csv(f"{data_dir_processed}{dataset}/{dataset}_processed_all.csv") 
tab_all.head()

,CaseID,Activity,time:timestamp,org:resource,org:role,case:Project,case:Task,case:OrganizationalEntity,case:Cost Type,case:RequestedAmount,case:Activity,case:RfpNumber
0,1,Request For Payment SUBMITTED by EMPLOYEE,0.000000,STAFF MEMBER,EMPLOYEE,project 148216,UNKNOWN,organizational unit 65463,0,34.336343,UNKNOWN,request for payment number 148215
1,1,Request For Payment FINAL_APPROVED by SUPERVISOR,3.761200,STAFF MEMBER,SUPERVISOR,project 148216,UNKNOWN,organizational unit 65463,0,34.336343,UNKNOWN,request for payment number 148215
2,1,Request For Payment REJECTED by MISSING,11.499992,STAFF MEMBER,MISSING,project 148216,UNKNOWN,organizational unit 65463,0,34.336343,UNKNOWN,request for payment number 148215
3,1,Request For Payment SUBMITTED by EMPLOYEE,15.337479,STAFF MEMBER,EMPLOYEE,project 148216,UNKNOWN,organizational unit 65463,0,34.336343,UNKNOWN,request for payment number 148215
4,1,Request For Payment APPROVED by PRE_APPROVER,15.337486,STAFF MEMBER,PRE_APPROVER,project 148216,UNKNOWN,organizational unit 65463,0,34.336343,UNKNOWN,request for payment number 148215


In [76]:
tab_train = pd.read_csv(f"{data_dir_processed}/{dataset}/{dataset}_processed_train.csv")
tab_valid = pd.read_csv(f"{data_dir_processed}/{dataset}/{dataset}_processed_valid.csv")
tab_test = pd.read_csv(f"{data_dir_processed}/{dataset}/{dataset}_processed_test.csv")

In [77]:
with open("dataset_features.json", 'r') as file:
    dataset_info = json.load(file)[dataset]

In [78]:
dataset_info

{'categorical': ['org:resource',
  'Activity',
  'org:role',
  'case:Project',
  'case:Task',
  'case:OrganizationalEntity',
  'case:Activity',
  'case:RfpNumber'],
 'numerical': ['time:timestamp', 'case:RequestedAmount']}

In [79]:
categorical_columns = dataset_info["categorical"]
real_value_columns = dataset_info["numerical"]

In [80]:
for k in categorical_columns:
    tab_all[k] = tab_all[k].astype("object")
    tab_train[k] = tab_train[k].astype("object")
    tab_valid[k] = tab_valid[k].astype("object")
    tab_test[k] = tab_test[k].astype("object")
    
 

In [81]:
from numpy import NaN
#if dataset == "sp2020":
#    tab_all["REPAIR_IN_TIME_5D"] = [float(x) if x is not NaN else x for x in tab_all["REPAIR_IN_TIME_5D"].values]
#    tab_train["REPAIR_IN_TIME_5D"] = [float(x) if x is not NaN else x for x in tab_train["REPAIR_IN_TIME_5D"].values]
#    tab_valid["REPAIR_IN_TIME_5D"] = [float(x) if x is not NaN else x for x in tab_valid["REPAIR_IN_TIME_5D"].values]
#    tab_test["REPAIR_IN_TIME_5D"] = [float(x) if x is not NaN else x for x in tab_test["REPAIR_IN_TIME_5D"].values]
    

In [82]:
for k in categorical_columns:
    print(f"{k} {tab_test[k].values.dtype}")

org:resource object
Activity object
org:role object
case:Project object
case:Task object
case:OrganizationalEntity object
case:Activity object
case:RfpNumber object


In [83]:
dataset

'BPI20_RequestForPayment'

In [84]:
from numpy import NaN
from math import log

def log_norm(x):
    return log(x+1)

if dataset == "BPI20_RequestForPayment_CZ":
    tab_all["case:RequestedAmount"] = tab_all["case:RequestedAmount"].apply(log_norm)
    tab_train["case:RequestedAmount"] = tab_train["case:RequestedAmount"].apply(log_norm)
    tab_valid["case:RequestedAmount"] = tab_valid["case:RequestedAmount"].apply(log_norm)
    tab_test["case:RequestedAmount"] = tab_test["case:RequestedAmount"].apply(log_norm)
   

### Prepare the graphs

In [85]:
from utils import get_case_ids, get_one_hot_encodings

from torch import tensor,int64, float32
from torch_geometric.data import HeteroData


In [86]:
import sklearn.preprocessing

def get_one_hot_encoder(dataset: pd.DataFrame, key: str):
    datas = dataset[key].unique()
    datas = datas.astype(np.str_)
    datas = datas.reshape([len(datas), 1])
    onehot = sklearn.preprocessing.OneHotEncoder()
    onehot.fit(datas)
    return onehot

In [87]:
categorical_columns

['org:resource',
 'Activity',
 'org:role',
 'case:Project',
 'case:Task',
 'case:OrganizationalEntity',
 'case:Activity',
 'case:RfpNumber']

In [88]:
ONE_HOT_ENCODERS = {k: get_one_hot_encoder(tab_all, k) for k in categorical_columns}
ONE_HOT_ENCODERS

{'org:resource': OneHotEncoder(),
 'Activity': OneHotEncoder(),
 'org:role': OneHotEncoder(),
 'case:Project': OneHotEncoder(),
 'case:Task': OneHotEncoder(),
 'case:OrganizationalEntity': OneHotEncoder(),
 'case:Activity': OneHotEncoder(),
 'case:RfpNumber': OneHotEncoder()}

In [89]:
def add_new_timestamp(trace: pd.DataFrame):
    times = list(trace["time:timestamp"].copy())
    for i in range(1,len(times)):
        times[i] = times[i] - times[0]
    times[0] = 0.
    trace2 = trace.copy()
    trace2["time:timestamp"] = times 
    return trace2

In [90]:
def get_node_features(trace: pd.DataFrame, cat_features, real_features) -> dict:
    columns_static = [c for c in trace if len(set(trace[c])) == 1]

    res = {}

    for key in trace:
        values = trace[key].values
        if key in cat_features:
            onehot_encoder = ONE_HOT_ENCODERS[key]
            values = values.astype(np.str_)
            if key not in columns_static:
                try:
                    res[key] = tensor(
                        get_one_hot_encodings(onehot_encoder, values),
                        dtype=float32,
                        requires_grad=True
                    )
                except ValueError:
                    print("Error in the encoding")
                    print(key)
                    print(values)
            else:
                res[key] = tensor(
                    get_one_hot_encodings(onehot_encoder, np.array([values[0]])),
                    dtype=float32,
                    requires_grad=True
                )
        if key in real_features:
            if key not in columns_static:
                res[key] = tensor(values,  dtype=float32,requires_grad=True)
            else:
                res[key] = tensor([values[0]], dtype=float32,requires_grad=True)
            res[key] = res[key].reshape(res[key].shape[0], 1)
        
    

    return res


In [91]:
def compute_edges_indexs(node_features: dict, prefix_len):
    res = {}
    keys = node_features.keys()
    indexes = [[i, i + 1] for i in range(prefix_len-1)]
    # activities indexes
    for k in keys:
        if len(node_features[k]) != 1:
            if k == "Activity":
                res[(k, "followed_by", k)] = indexes
                for k2 in keys:
                    if k2 != k:
                        if len(node_features[k2]) == 1:
                            res[(k, "related_to", k2)] = [
                                [i, 0] for i in range(prefix_len)
                            ]
                        else:
                            res[(k, "related_to", k2)] = [
                                [i, i] for i in range(prefix_len)
                            ]
            else:
                res[(k, "related_to", k)] = indexes

    return res

In [92]:
from tqdm.notebook import tqdm

In [93]:

from copy import copy
import torch

from torch_geometric.transforms import ToUndirected
UNDIRECT_TRANSFORMATION = ToUndirected()


def build_prefixes_graph_from_trace(trace, cat_features, real_features):
    X = []  # graphs
   
    trace = add_new_timestamp(trace)
    
    node_features = get_node_features(trace, cat_features, real_features)
    
    for prefix in range(1, len(trace)-1):
        
        G = HeteroData()
        for k in node_features:
            G[k].x = node_features[k][:(prefix+1)]

        edges_indexes = compute_edges_indexs(node_features, prefix_len=prefix+1)

        for k in edges_indexes:
            ce = [[], []]
            for i in range(len(edges_indexes[k])):
                ce[0].append(edges_indexes[k][i][0])
                ce[1].append(edges_indexes[k][i][1])
            edges_indexes[k] = ce

        for k in edges_indexes:
            G[k].edge_index = tensor(edges_indexes[k], dtype=torch.long)

        G.y = {}
        for k in node_features:
            if len(node_features[k]) != 1:
                if k in cat_features:
                    G.y[k] = torch.max(node_features[k][prefix+1], 0)[1].reshape(1,-1)[0].detach().clone()
                else:
                    G.y[k] = node_features[k][prefix+1].reshape(1,-1)[0].detach().clone()
            else:
                if k in cat_features:
                    G.y[k] = torch.max(node_features[k][0], 0)[1].reshape(1,-1)[0].detach().clone()
                else:
                    G.y[k] = node_features[k][0].reshape(1,-1)[0].detach().clone()
        
        G = UNDIRECT_TRANSFORMATION(G)
             
        X.append(G)
        
    return X

## Create the graph datasets

In [94]:
case_train_ids = get_case_ids(tab_train)
case_valid_ids = get_case_ids(tab_valid)
case_test_ids = get_case_ids(tab_test)

In [95]:
print(len(case_train_ids))
print(len(case_valid_ids))
print(len(case_test_ids))

4131
1377
1378


In [96]:
tab_train["CaseID"] = tab_train["CaseID"].astype(np.str_)
tab_valid["CaseID"] = tab_valid["CaseID"].astype(np.str_)
tab_test["CaseID"] = tab_test["CaseID"].astype(np.str_)

In [97]:
if not os.path.isdir(f"{data_dir_graphs}{dataset}"):
    os.mkdir(f"{data_dir_graphs}{dataset}")

In [98]:
from tqdm.notebook import tqdm

In [99]:
print("Preparing training dataset...")

X_train = []

for i in tqdm(range(len(case_train_ids))):
    trace = (
        tab_train.query(f"CaseID == '{case_train_ids[i]}'")
        .reset_index()
        .drop(columns="index")
        .drop(columns="CaseID")
    )

    if len(trace) > 2:
        graphs = build_prefixes_graph_from_trace(
            trace=trace,
            cat_features=categorical_columns,
            real_features=real_value_columns,
        )
        
        N_GRAPHS = len(graphs)
        
        for j in range(N_GRAPHS):
            X_train.append(graphs[j])


#print(postman_constraints_training)

breakpoint()

torch.save(X_train, f"{data_dir_graphs}{dataset}/train_set.pt")

print("Train Graphs created!\n\n")

Preparing training dataset...


  0%|          | 0/4131 [00:00<?, ?it/s]

Train Graphs created!




In [100]:
del X_train

In [101]:
print("Preparing validation dataset...")

X_val = []

for i in tqdm(range(len(case_valid_ids))):
    trace = (
        tab_valid.query(f"CaseID == '{case_valid_ids[i]}'")
        .reset_index()
        .drop(columns="index")
        .drop(columns="CaseID")
    )

    if len(trace) > 2:
        graphs = build_prefixes_graph_from_trace(
            trace=trace,
            cat_features=categorical_columns,
            real_features=real_value_columns,
        )
        
        N_GRAPHS = len(graphs)

        
        for j in range(N_GRAPHS):
            X_val.append(graphs[j])



torch.save(X_val, f"{data_dir_graphs}{dataset}/validation_set.pt")
print("Val Graph created!\n\n")

Preparing validation dataset...


  0%|          | 0/1377 [00:00<?, ?it/s]

Val Graph created!




In [102]:
data_dir_graphs

'/home/sebastiano.dissegna/SephigraphV2/data/datasets/graphs/'

In [103]:
del X_val

In [104]:
print("Preparing test dataset...")

X_test = []


for i in tqdm(range(len(case_test_ids))):
    trace = (
        tab_test.query(f"CaseID == '{case_test_ids[i]}'")
        .reset_index()
        .drop(columns="index")
        .drop(columns="CaseID")
    )

    if len(trace) > 2:
        graphs = build_prefixes_graph_from_trace(
            trace=trace,
            cat_features=categorical_columns,
            real_features=real_value_columns,
        )
        
        N_GRAPHS = len(graphs)
        
        for j in range(N_GRAPHS):
            X_test.append(graphs[j])
            

torch.save(X_test, f"{data_dir_graphs}{dataset}/test_set.pt")
print("Test Graphs created!\n\n")

Preparing test dataset...


  0%|          | 0/1378 [00:00<?, ?it/s]

Test Graphs created!




In [105]:
del X_test